# How Agents Work

## Abstract

What turns a language-model call into an agent? This notebook answers by placing the same model and the same broken Python repository inside progressively richer control systems. Tools expand what a model can do. Agency begins when the model chooses its next action from the latest observation, within boundaries enforced by code.


[Open in Kaggle](https://www.kaggle.com/code/addarm/how-agents-work)

![Banner](https://raw.githubusercontent.com/adamd1985/quant_research/refs/heads/main/images/agents_how_they_work_banner.png)

## Introduction

An agentic system places a model inside a runtime that provides state, tools, observations, and boundaries. It becomes an agent when the model chooses its next action after observing prior results. That single distinction is what this article is about.

We build a small runtime around a deliberately broken Python repository, then read its behaviour off traces, tests, token counts, and cost records. Keep two words apart from the start. The runtime is the harness: it supplies tools, executes requested actions, returns observations, and enforces limits. The agent is the model acting inside that harness. The harness defines the available actions, limits, and stopping conditions. The model chooses its next action within those boundaries.

## Why this matters now

"Agent" has become a loose label. Anthropic's *Building Effective Agents* draws a sharper line: a workflow sends models and tools along paths written in code, while an agent lets the model direct its own actions and choose how to use tools. The difference is who picks the next step after seeing the latest result.

Fixed Python control flow is easy to reproduce, budget, and test. Model-directed control can adapt when the next useful step depends on an observation, but it adds variance, latency, and new failure modes. Real systems usually mix the two: code sets the boundaries, the model chooses within them. The rest of this notebook walks that spectrum, from a plain call that can touch nothing to a bounded autonomous agent that edits code and proves its work with tests.

### Learning objectives

By the end, you should be able to:

1. distinguish a plain call, an augmented call, a workflow, and an agent;
2. recognise five common multi-call execution topologies;
3. explain why validation, budgets, approval, and evaluation belong in the runtime.


## Notebook Setup

The examples use OpenRouter, although the same harness could call a direct API, Vertex AI, Bedrock, or another provider. Set `OPENROUTER_API_KEY` before running the notebook. The experiments make paid model calls and use a fixed seed where the provider supports it; model outputs can still vary.


In [1]:
import ast
import hashlib
import itertools
import json
import os
import platform
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import threading
import time
import urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from jsonschema import Draft202012Validator

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1").rstrip("/")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")
SEED = 42
PROVIDER_POLICY = {"require_parameters": True, "data_collection": "deny"}
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is required for this notebook")
CALL_LOG, _CALL_COUNTER, _LOG_LOCK = [], itertools.count(1), threading.Lock()


def strict(props, required=None):
    return {
        "type": "object",
        "properties": props,
        "required": required or list(props),
        "additionalProperties": False,
    }

In [2]:
def call_model(
    messages,
    pattern,
    tools=None,
    tool_choice=None,
    response_format=None,
    temperature=0,
    max_tokens=2000,
):
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "seed": SEED,
        "provider": PROVIDER_POLICY,
        "usage": {"include": True},
    }
    for key, val in (
        ("tools", tools),
        ("tool_choice", tool_choice),
        ("response_format", response_format),
    ):
        if val is not None:
            payload[key] = val
    req = urllib.request.Request(
        OPENROUTER_BASE_URL + "/chat/completions",
        data=json.dumps(payload).encode(),
        method="POST",
        headers={
            "Authorization": "Bearer " + OPENROUTER_API_KEY,
            "Content-Type": "application/json",
            "X-Title": "How Agents Work",
        },
    )
    with _LOG_LOCK:
        call_id = f"c{next(_CALL_COUNTER):04d}"
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            body = json.loads(resp.read().decode())
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f"OpenRouter call failed: {exc.read().decode(errors='replace')}") from exc
    choice = body["choices"][0]
    msg = choice["message"]
    usage = body.get("usage", {}) or {}
    rec = {
        "call_id": call_id,
        "pattern": pattern,
        "actual_model": body.get("model"),
        "latency_s": round(time.perf_counter() - started, 3),
        "content": msg.get("content"),
        "tool_calls": msg.get("tool_calls", []) or [],
        "tokens": usage.get("total_tokens"),
        "cost": usage.get("cost"),
        "error": None,
    }
    with _LOG_LOCK:
        CALL_LOG.append(rec)
    return rec

In [3]:
def structured_format(name, schema):
    return {
        "type": "json_schema",
        "json_schema": {"name": name, "strict": True, "schema": schema},
    }


def call_structured(messages, pattern, name, schema):
    rec = call_model(messages, pattern, response_format=structured_format(name, schema))
    value = json.loads(rec["content"] or "")
    errs = list(Draft202012Validator(schema).iter_errors(value))
    if errs:
        raise ValueError(f"structured response failed local validation: {errs[0].message}")
    return value, rec


def summarize_calls(pattern):
    rows = [r for r in CALL_LOG if r["pattern"] == pattern]
    return {
        "calls": len(rows),
        "tokens": sum((r["tokens"] or 0) for r in rows),
        "cost": round(sum((r["cost"] or 0) for r in rows), 6),
    }


print({"model": OPENROUTER_MODEL, "python": platform.python_version(), "seed": SEED})

{'model': 'openai/gpt-4.1-mini', 'python': '3.12.12', 'seed': 42}


## Data / Simulation

The "data" is a tiny synthetic repository with two source files and two protected test files. Four defects are planted deliberately. Rebuilding this repository before every experiment gives each pattern the same starting point and prevents one run from leaking changes into the next.


In [4]:
BASELINE_FILES = {
    "pricing.py": "def percentage_discount(price, percent):\n    return round(price - percent, 2)\n\n\ndef shipping_cost(subtotal):\n    return 0.0 if subtotal > 50 else 5.0\n",
    "orders.py": 'def can_cancel(status):\n    if status in {"pending", "processing"}:\n        return True\n    if status == "shipped":\n        return False\n    return True\n',
    "tests/test_pricing.py": "import pytest\n\nfrom pricing import percentage_discount, shipping_cost\n\n\ndef test_percentage_discount():\n    assert percentage_discount(200, 20) == 160\n\n\ndef test_free_shipping_threshold():\n    assert shipping_cost(49.99) == 5.0\n    assert shipping_cost(50.00) == 0.0\n\n\ndef test_pricing_input_validation():\n    with pytest.raises(ValueError):\n        percentage_discount(-1, 10)\n    with pytest.raises(ValueError):\n        percentage_discount(10, 101)\n",
    "tests/test_orders.py": 'import pytest\n\nfrom orders import can_cancel\n\n\ndef test_order_status_edge_cases():\n    assert can_cancel("pending") is True\n    assert can_cancel("processing") is True\n    assert can_cancel("shipped") is False\n    assert can_cancel("delivered") is False\n    with pytest.raises(ValueError):\n        can_cancel("unknown")\n',
}
WORKSPACE = Path(tempfile.mkdtemp(prefix="agents_"))
PROJECT_ROOT = WORKSPACE / "mini_project"
SOURCE_ALLOWLIST = ("pricing.py", "orders.py")
TEST_PATHS = ("tests/test_pricing.py", "tests/test_orders.py")
PYTEST_ENV = {**os.environ, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}


def run_process(command, timeout=30, env=None):
    done = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout,
        env=env,
    )
    return {"exit_code": done.returncode, "stdout": done.stdout, "stderr": done.stderr}


def safe_path(relative):
    if not isinstance(relative, str) or not relative:
        raise ValueError("path must be a non-empty string")
    root = PROJECT_ROOT.resolve()
    candidate = (PROJECT_ROOT / relative).resolve()
    if candidate != root and root not in candidate.parents:
        raise ValueError("path escapes mini_project")
    return candidate


def test_hashes():
    return {n: hashlib.sha256(safe_path(n).read_bytes()).hexdigest()[:12] for n in TEST_PATHS}


def snapshot():
    return {n: safe_path(n).read_text(encoding="utf-8") for n in SOURCE_ALLOWLIST}


def run_tests():
    return run_process([sys.executable, "-m", "pytest", "-q", "--color=no"], timeout=60, env=PYTEST_ENV)


def gather_evidence():
    return {"files": snapshot(), "tests": run_tests()["stdout"]}

In [5]:
def failure_count(result):
    m = re.search(r"(\d+) failed", result["stdout"])
    return int(m.group(1)) if m else 0

In [6]:
def reset_project():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT, onexc=lambda fn, p, _e: (os.chmod(p, stat.S_IWRITE), fn(p)))
    for rel, content in BASELINE_FILES.items():
        target = PROJECT_ROOT / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")


reset_project()
CANONICAL_TESTS = test_hashes()


def start_experiment(name, expected_failures=4):
    reset_project()
    assert test_hashes() == CANONICAL_TESTS, f"{name}: protected test files drifted"
    seen = failure_count(run_tests())
    assert seen == expected_failures, f"{name}: expected {expected_failures} failing, saw {seen}"


start_experiment("baseline")
print("baseline failing tests:", failure_count(run_tests()))

baseline failing tests: 4


The four tests expose four planted defects: `percentage_discount` subtracts a flat amount instead of a percentage, `shipping_cost` uses a strict `>` that mishandles the 50.00 boundary, `can_cancel` returns `True` for a delivered order, and neither pricing function validates its inputs. Four tests fail, and that count is the invariant we assert at the start of every experiment. `start_experiment` rebuilds the repository from the same baseline, then checks that the protected test files are byte-for-byte unchanged and that exactly four tests fail. No experiment inherits a mutation or a leftover fix from the one before it. We only bring Git into the picture for the approval demo, where a commit is the action under control.

The layer map below sets the order for the notebook.

## Methods / Theory


### Four layers of an agent system

The zoo of agent patterns reads more cleanly as four stacked layers, each answering a different question.

1. The **capability layer** asks what a single model call can reach. On its own it only transforms text. Give it retrieval, a scratch memory, or tools and you widen that reach. Adding read-only tools does not by itself make a system an agent: a call that can read a file is still a single augmented call.

2. The **control-flow layer** asks who chooses the next step. If your Python code fixes the order, that's a workflow. If the model picks the next action from what it just observed, and keeps doing so, that's an agent in Anthropic's sense.

3. The **execution-topology layer** asks what shape the control flow takes once several calls are in play: a fixed chain, a router, a fan-out, an orchestrator that plans its own workers, or a generate-and-check loop.

4. The **ownership layer** asks who owns the final answer when specialists get involved, and what controls (validation, budgets, tracing, approval, sandboxing) keep the whole thing bounded.


```
Layer 1  Capability        plain call  ->  + context / memory / tools (augmented)
Layer 2  Control flow      workflow (Python decides)  vs  agent (model decides next step)
Layer 3  Topology          chaining -> routing -> parallelization -> orchestrator-workers -> evaluator-optimizer
Layer 4  Ownership+control  manager with specialists / handoff  +  validation, budgets, tracing, approval, sandbox
```

### The action-observation loop

Let $s_t$ be the state at model turn $t$: the conversation so far plus the relevant environment state. A controller $\pi$ chooses an action from that state, the environment executes an allowed action and returns an observation, and the observation folds into the next state:

$$a_t = \pi(s_t), \qquad o_t = \operatorname{env}(s_t, a_t), \qquad s_{t+1} = f(s_t, a_t, o_t).$$

Here $a_t$ is the chosen action, either a tool call or a decision to stop; $o_t$ is what the environment returns after running it; $\operatorname{env}$ is the environment's response function; and $f$ is the update that appends the action and its observation to the state. In plain words: look at the current evidence, choose an action, run it, read the result, repeat. The only thing that changes between a workflow and an agent is who plays the part of $\pi$.

The notation is deliberately close to reinforcement learning, because many of these patterns borrow from it, and the loop echoes ReAct's interleaving of reasoning and acting (Yao et al., 2023). We do not claim to read the model's private reasoning. What we record is the observable action-observation loop: proposed tools, validated arguments, returned results, costs, and the decision to stop. A production runtime needs more than one budget, because model turns, tool calls, writes, subprocesses, wall-clock time, tokens, and money all grow at different rates, so the loop below tracks turn, tool-call, time, token, and cost limits separately. These are application controls, not isolation.


### Assumptions and scope

The repository is local, the tests are deterministic, tool arguments are JSON, and each experiment starts from the same files. A single successful run is a worked example, not a reliability estimate. Provider routing and model sampling may change the exact trace.


### A small, bounded runtime

The next cells define the runtime in layers: tool descriptions, safe file operations, argument validation, one model-directed loop, and a compact trace. These helpers are infrastructure; the experiments that follow are the lesson.


In [7]:
TOOL_SPECS = {
    "list_files": ("List the Python files in mini_project.", {}, []),
    "read_file": (
        "Read one UTF-8 file from mini_project.",
        {"path": {"type": "string"}},
        ["path"],
    ),
    "search_code": (
        "Search all Python files for a substring.",
        {"query": {"type": "string"}},
        ["query"],
    ),
    "run_tests": ("Run pytest and return stdout, stderr, exit code.", {}, []),
    "apply_patch": (
        "Replace the first exact old string in pricing.py or orders.py.",
        {
            "path": {"enum": list(SOURCE_ALLOWLIST)},
            "old": {"type": "string"},
            "new": {"type": "string"},
        },
        ["path", "old", "new"],
    ),
    "write_file": (
        "Write a complete pricing.py or orders.py file.",
        {
            "path": {"enum": list(SOURCE_ALLOWLIST)},
            "content": {"type": "string"},
        },
        ["path", "content"],
    ),
}


def tool_schema(name):
    description, properties, required = TOOL_SPECS[name]
    parameters = strict(properties, required)
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }

In [8]:
def _writable(path):
    if path not in SOURCE_ALLOWLIST:
        raise PermissionError(f"writes restricted to {SOURCE_ALLOWLIST}; refused '{path}'")
    target = PROJECT_ROOT / path
    if target.is_symlink() or (target.exists() and target.resolve() != safe_path(path)):
        raise PermissionError(f"refused symlink '{path}'")
    return safe_path(path)


def list_files():
    paths = sorted(PROJECT_ROOT.rglob("*.py"))
    return [str(path.relative_to(PROJECT_ROOT)).replace("\\", "/") for path in paths]


def read_file(path):
    content = safe_path(path).read_text(encoding="utf-8")
    return {"path": path, "content": content}


def search_code(query):
    if not isinstance(query, str) or not query:
        raise ValueError("query must be a non-empty string")

    hits = []
    for relative_path in list_files():
        lines = safe_path(relative_path).read_text(encoding="utf-8").splitlines()
        for line_number, line in enumerate(lines, start=1):
            if query.lower() in line.lower():
                hits.append({"path": relative_path, "line": line_number, "text": line})
    return hits


def apply_patch(path, old, new):
    target = _writable(path)
    current = target.read_text(encoding="utf-8")
    if old not in current:
        return {"path": path, "changed": False, "duplicate": new in current}

    updated = current.replace(old, new, 1)
    ast.parse(updated, filename=path)
    target.write_text(updated, encoding="utf-8")
    return {"path": path, "changed": True}

In [9]:
def write_file(path, content):
    target = _writable(path)
    ast.parse(content, filename=path)
    if target.exists() and target.read_text(encoding="utf-8") == content:
        return {"path": path, "changed": False}

    target.write_text(content, encoding="utf-8")
    return {"path": path, "changed": True}


def apply_edits(edits):
    applied = []
    rejected = []
    for edit in edits:
        try:
            result = apply_patch(**edit)
            applied.append({"path": edit.get("path"), "changed": result["changed"]})
        except Exception as exc:
            rejected.append(
                {
                    "path": edit.get("path"),
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
    return {"applied": applied, "rejected": rejected}


TOOL_REGISTRY = {
    "list_files": list_files,
    "read_file": read_file,
    "search_code": search_code,
    "run_tests": run_tests,
    "apply_patch": apply_patch,
    "write_file": write_file,
}
READ_ONLY = ["list_files", "read_file", "search_code", "run_tests"]
AUTONOMOUS_TOOLS = [*READ_ONLY, "write_file"]
ACTIVE_AGENT = ["controller"]


def validate_tool_call(name, arguments):
    if name not in TOOL_REGISTRY:
        raise ValueError(f"unknown tool: {name}")

    parameters = tool_schema(name)["function"]["parameters"]
    errors = list(Draft202012Validator(parameters).iter_errors(arguments))
    if errors:
        raise ValueError(errors[0].message)
    if "path" in arguments:
        safe_path(arguments["path"])

In [10]:
def execute_tool_call(tool_call):
    name = tool_call["function"]["name"]
    try:
        arguments = json.loads(tool_call["function"]["arguments"])
        validate_tool_call(name, arguments)
        result = TOOL_REGISTRY[name](**arguments)
        succeeded = result.get("exit_code") == 0 if name == "run_tests" else True
    except Exception as exc:
        arguments = {}
        result = {"error": f"{type(exc).__name__}: {exc}"}
        succeeded = False
    return name, arguments, result, succeeded


def summarize_tool_result(name, result):
    if name == "run_tests" and "stdout" in result:
        output_lines = result["stdout"].strip().splitlines()
        final_line = output_lines[-1] if output_lines else ""
        return f"exit={result['exit_code']} {final_line}"
    if isinstance(result, dict) and "content" not in result:
        return json.dumps(result)[:80]
    return "ok"


def agent_outcome(status, answer, messages, trace, tool_calls, tokens, cost):
    usage = {
        "turns": len({row["step"] for row in trace}),
        "tool_calls": tool_calls,
        "tokens": tokens,
        "cost": round(cost, 6),
    }
    return {
        "status": status,
        "answer": answer,
        "trace": trace,
        "messages": messages,
        "usage": usage,
    }

In [11]:
def append_tool_observation(tool_call, step, messages, trace, seen_actions):
    name, arguments, result, succeeded = execute_tool_call(tool_call)
    fingerprint = (name, json.dumps(arguments, sort_keys=True))
    repeated = fingerprint in seen_actions
    seen_actions.add(fingerprint)

    argument_label = arguments.get("path") or arguments.get("query") or ""
    trace.append(
        {
            "step": step,
            "agent": ACTIVE_AGENT[-1],
            "action": name,
            "args": str(argument_label)[:40],
            "result": summarize_tool_result(name, result)[:80],
            "ok": succeeded,
            "changed": bool(result.get("changed")),
            "repeated": repeated,
        }
    )
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "name": name,
            "content": json.dumps(result)[:4000],
        }
    )


def append_stop(trace, step, content):
    trace.append(
        {
            "step": step,
            "agent": ACTIVE_AGENT[-1],
            "action": "stop",
            "args": "",
            "result": "final response",
            "ok": bool(content),
        }
    )

In [12]:
def run_agent(
    messages,
    tool_names,
    pattern,
    budget=12,
    max_tool_calls=32,
    max_elapsed_s=180,
    max_tokens=50_000,
    max_cost=0.25,
):
    tools = [tool_schema(name) for name in tool_names]
    messages = list(messages)
    trace = []
    seen_actions = set()
    started = time.perf_counter()
    tool_calls = 0
    tokens = 0
    cost = 0.0

    def finish(status, answer=None):
        return agent_outcome(status, answer, messages, trace, tool_calls, tokens, cost)

    for step in range(budget):
        if time.perf_counter() - started >= max_elapsed_s:
            return finish("time_budget_exhausted")

        response = call_model(
            messages,
            pattern,
            tools=tools,
            tool_choice="auto",
        )
        tokens += response["tokens"] or 0
        cost += response["cost"] or 0
        messages.append(
            {
                "role": "assistant",
                "content": response["content"],
                "tool_calls": response["tool_calls"],
            }
        )

        if tokens > max_tokens:
            return finish("token_budget_exhausted")
        if cost > max_cost:
            return finish("cost_budget_exhausted")
        if not response["tool_calls"]:
            append_stop(trace, step, response["content"])
            return finish("completed", response["content"])

        for tool_call in response["tool_calls"]:
            if tool_calls >= max_tool_calls:
                return finish("tool_budget_exhausted")
            tool_calls += 1
            append_tool_observation(
                tool_call,
                step,
                messages,
                trace,
                seen_actions,
            )

    return finish("turn_budget_exhausted")

In [13]:
def show_trace(trace):
    columns = ["step", "agent", "action", "args", "result"]
    print(pd.DataFrame(trace)[columns].to_string(index=False))

`run_agent` is our small harness. Harness is a term you'll meet constantly around agents; it just means the runtime with its controls. This one validates tool arguments, executes allowed tools, returns observations, records a trace, and stops when the model finishes or a budget is exhausted. The same runtime supports the model-directed examples below, while Python defines the surrounding workflow topologies.

The implementation stays deliberately small. It truncates tool observations at 4,000 characters and keeps state in memory. It has no retries, durable checkpoint, cancellation protocol, context compaction, or operating-system sandbox.

## Experiments / Results

Each experiment follows the same pattern:

1. rebuild the broken repository;
2. run one control pattern;
3. measure observable behaviour;
4. record whether the declared criterion was met.


### Capability: plain and augmented calls


#### The plain call

The simplest capability layer is a bare call with no tools and no environment interface.

We name the repository in the prompt and ask for a fix. The measurable result is that the source files and the pytest exit code do not change, because the model had no way to reach them. If the model answers by explaining that it cannot access local files, that is the correct response to an impossible request, not a hallucination. A hallucination would be a confident claim to have edited files it never saw.


In [14]:
start_experiment("plain call")
plain_before = snapshot()
plain = call_model(
    [
        {
            "role": "user",
            "content": "Fix every bug in the local mini_project repository and report the pytest result.",
        }
    ],
    "plain call",
)
plain_tests = run_tests()
print("MODEL SAID (first 400 chars):")
print((plain["content"] or "")[:400])
plain_obs = {
    "files_unchanged": snapshot() == plain_before,
    "test_exit_code": plain_tests["exit_code"],
}
print("\nMEASURABLE RESULT:", json.dumps(plain_obs))
plain_ok = plain_obs["files_unchanged"] and plain_tests["exit_code"] != 0
assert plain_ok

MODEL SAID (first 400 chars):
I don't have direct access to your local files or repositories. However, you can share the code or error messages from your mini_project repository here, and I can help you identify and fix the bugs.

Alternatively, you can run the following commands in your local environment to check for bugs and run tests:

1. Navigate to your project directory:
```bash
cd path/to/mini_project
```

2. Run pytest

MEASURABLE RESULT: {"files_unchanged": true, "test_exit_code": 1}


The repository is untouched and the suite still fails. Whatever the model wrote, it was words about code, not a change to code, and text alone moves nothing. Everything agentic from here is about closing the gap between a message and an effect, one controlled tool at a time.

#### An augmented call

Now we augment the call, and it still will not be an agent. Python gathers a fixed bundle of evidence (both source files and the pytest output) and hands it to a single structured decision. The model diagnoses; it does not choose what to look at next and it has no write tool.

This is deliberately not the open-ended loop we'll later call an agent. It is one call made more capable by context, which is what "augmented LLM" should mean. The criterion is stricter than "returned an issue": the diagnosis has to name the percentage defect and the missing input validation, and the source has to stay byte-for-byte unchanged.


In [15]:
DIAGNOSIS_SCHEMA = strict(
    {
        "issues": {
            "type": "array",
            "items": strict(
                {
                    "path": {"enum": ["pricing.py", "orders.py"]},
                    "problem": {"type": "string"},
                    "evidence": {"type": "string"},
                }
            ),
        }
    }
)


def diagnosis_covers_expected(text):
    low = text.lower()
    functions = all(name in low for name in ["percentage_discount", "shipping_cost", "can_cancel"])
    validation = any(term in low for term in ["input validation", "validate", "valueerror"])
    return functions and validation


start_experiment("augmented LLM")
before = snapshot()
diagnosis, _ = call_structured(
    [
        {
            "role": "system",
            "content": "Diagnose every failing test using only the supplied files and pytest output. Name each affected function and any missing input validation. Do not invent files.",
        },
        {"role": "user", "content": json.dumps(gather_evidence(), sort_keys=True)},
    ],
    "augmented LLM",
    "diagnosis",
    DIAGNOSIS_SCHEMA,
)
augmented_covers = diagnosis_covers_expected(json.dumps(diagnosis))
augmented_obs = {
    "issues_found": len(diagnosis["issues"]),
    "covers_all_defects": augmented_covers,
    "files_unchanged": snapshot() == before,
}
print("\nMEASURABLE RESULT:", json.dumps(augmented_obs))
augmented_ok = augmented_obs["files_unchanged"] and augmented_covers
assert augmented_ok, augmented_obs


MEASURABLE RESULT: {"issues_found": 4, "covers_all_defects": true, "files_unchanged": true}


The model returned a structured diagnosis grounded in the evidence we chose for it, and the source files are unchanged. Read-only augmentation made the call more useful without making it agentic. Python still decided everything about what the model saw and when it stopped. The next layer hands some of that decision-making across.

### Control flow: workflow or agent?

Now the control-flow question made concrete. Both runs get the same broken repository from a fresh baseline and the same goal: locate the defects. The workflow follows a sequence we wrote (list, read, read, run tests, diagnose). The agent gets the read-only tools and decides each step itself. Neither can write, which is the point: agency is about who chooses the next action, not about write access.

Watch the difference in who is in charge, and compare the call count and cost. The agent must take at least one valid tool action, complete the task, and ground its answer in the affected functions.


In [16]:
start_experiment("workflow")
workflow_actions = [
    "list_files",
    "read_file:pricing",
    "read_file:orders",
    "run_tests",
    "diagnose",
]
workflow_sources = {n: read_file(n) for n in SOURCE_ALLOWLIST}
workflow_tests = run_tests()
workflow_diagnosis, _ = call_structured(
    [
        {
            "role": "system",
            "content": "Diagnose every failing test in the supplied source and pytest output. Name each affected function and any missing input validation.",
        },
        {
            "role": "user",
            "content": json.dumps(
                {
                    "files": {n: workflow_sources[n]["content"] for n in SOURCE_ALLOWLIST},
                    "tests": workflow_tests["stdout"],
                },
                sort_keys=True,
            ),
        },
    ],
    "workflow",
    "diagnosis",
    DIAGNOSIS_SCHEMA,
)
workflow_ok = len(workflow_diagnosis["issues"]) >= 1 and diagnosis_covers_expected(json.dumps(workflow_diagnosis))
start_experiment("agent")
ACTIVE_AGENT.append("diagnoser")
agent_run = run_agent(
    [
        {
            "role": "system",
            "content": "Diagnose mini_project with the read-only tools. Decide each step yourself. Stop when you can name every defect with evidence, and in your final answer name each affected function and note any missing input validation.",
        },
        {"role": "user", "content": "Find every defect and cite the evidence."},
    ],
    READ_ONLY,
    "agent",
    budget=8,
)
ACTIVE_AGENT.pop()
agent_actions = [e["action"] for e in agent_run["trace"]]
successful_agent_tools = [e for e in agent_run["trace"] if e["action"] in READ_ONLY and e.get("ok")]

AttributeError: 'list' object has no attribute 'get'

In [ ]:
agent_answer = agent_run.get("answer") or ""
agent_ok = agent_run["status"] == "completed" and len(successful_agent_tools) >= 1 and diagnosis_covers_expected(agent_answer)
print("WORKFLOW (fixed by Python):", workflow_actions, "->", summarize_calls("workflow"))
print("AGENT (chosen by model)   :", agent_actions, "->", summarize_calls("agent"))
print("workflow grounded:", workflow_ok, "| agent completed+grounded:", agent_ok)

The workflow's action list is identical to what we typed, and its cost is knowable before you run it. The agent's list came out of the model, so it varies with the model's reading of the situation: it might stop early once confident, or spend an extra call double-checking. That flexibility is the whole appeal and the whole risk. A workflow can waste a step it didn't need; an agent can choose a step you didn't want. When the task is predictable, the workflow's repeatability usually wins. When the next useful step genuinely depends on what the last one revealed, you start paying for an agent.

In [ ]:
RESULTS = {}
EXPECTED = {
    "plain call": "No environment access; files and tests unchanged.",
    "augmented LLM": "Read-only diagnosis names the percentage defect and the missing validation; source unchanged.",
    "workflow": "Python-fixed sequence yields a diagnosis grounded in the affected functions.",
    "agent": "Model chooses read-only steps, completes, and grounds its answer in the affected functions.",
    "autonomous agent": "Multi-turn loop makes pytest pass with tests unchanged and no unauthorized files.",
    "prompt chaining": "Diagnose->patch->review with a deterministic gate; outcome is repaired or partial_repair, reported separately.",
    "routing": "Each request routed correctly and the handler proves fulfilment (specific test, asserting call, real symbols).",
    "parallelization": "Three independent reviews run concurrently with unique call ids.",
    "orchestrator-workers": "Constrained plan: Python fixes file ownership, model writes objectives; workers cite their files; synthesis names both.",
    "evaluator-optimizer": "Tests gate correctness; a rubric aggregate computed in Python gates quality; loop stops when both pass.",
    "manager-specialists": "Manager consults a bounded, non-repeated subset and owns the answer; a repeated consult fails the criterion.",
    "handoff": "Triage transfers ownership; specialist acts validly from the first step and grounds its diagnosis in both functions.",
    "runtime controls": "Allowed source write succeeds; protected-test, new-file, escape, and symlink writes are rejected.",
    "approval": "Simulated human approval; stale or modified diffs rejected; replay idempotent.",
}


def record_result(pattern, observed, criterion_met, note=""):
    RESULTS[pattern] = {
        "pattern": pattern,
        "expected": EXPECTED[pattern],
        "observed": observed,
        "criterion_met": bool(criterion_met),
        "note": note,
        **summarize_calls(pattern),
    }


record_result("plain call", plain_obs, plain_ok)
record_result("augmented LLM", augmented_obs, augmented_ok)
record_result(
    "workflow",
    {
        "actions": workflow_actions,
        "issues": len(workflow_diagnosis["issues"]),
        "grounded": workflow_ok,
    },
    workflow_ok,
)
record_result(
    "agent",
    {"actions": agent_actions, "status": agent_run["status"], "grounded": agent_ok},
    agent_ok,
)
print("registered", len(EXPECTED), "expected behaviours;", len(RESULTS), "recorded so far")

### The autonomous coding agent

This is the practical payoff of everything so far. We hand the model the loop. It gets the task and the symptom, the failing pytest output, and nothing else: no list of the four defects, no hint about the fix.

It must inspect the source and tests, decide what to change, edit only the allowed files, run the suite, react to what it sees, and stop only after it has observed a passing run. The criterion is strict: a genuine multi-turn trajectory, at least one write, a passing `run_tests` observation inside the trace, a final green suite, the protected tests untouched, no unauthorized file, and a non-empty final answer. Watch the observable trace do the work.


In [ ]:
start_experiment("autonomous agent")
auto_tests_before = test_hashes()
symptom = run_tests()["stdout"]
ACTIVE_AGENT.append("coder")
auto = run_agent(
    [
        {
            "role": "system",
            "content": (
                "You are a coding agent working in mini_project. You can read files, search, run the tests, and replace pricing.py or orders.py "
                "with write_file, which takes the complete new file contents. Tests are read-only. Inspect the code and the failing tests, then rewrite "
                "the source files so pytest passes, running the suite to check your work and reacting to each result. Do not claim completion until a "
                "run_tests observation shows exit code 0. Do not repeat an unchanged action."
            ),
        },
        {
            "role": "user",
            "content": "The suite is failing. Here is the current pytest output. Make it pass.\n\n" + symptom,
        },
    ],
    AUTONOMOUS_TOOLS,
    "autonomous agent",
    budget=16,
)
ACTIVE_AGENT.pop()
auto_final = run_tests()
unauthorized = set(list_files()) - set(SOURCE_ALLOWLIST) - set(TEST_PATHS)
cycles = len({e["step"] for e in auto["trace"]})
observed_green = any(r["action"] == "run_tests" and r.get("ok") is True for r in auto["trace"])
writes = sum(1 for r in auto["trace"] if r["action"] == "write_file" and r.get("ok") and r.get("changed"))
auto_obs = {
    "status": auto["status"],
    "cycles": cycles,
    "tool_calls": auto["usage"]["tool_calls"],
    "write_actions": writes,
    "observed_green_test": observed_green,
    "final_exit_code": auto_final["exit_code"],
    "tests_unchanged": auto_tests_before == test_hashes(),
    "unauthorized_files": sorted(unauthorized),
}
print("\nOBSERVABLE TRACE:")
show_trace(auto["trace"])
print("\nMEASURABLE RESULT:", json.dumps(auto_obs))
AUTO_FIXED = snapshot()

In [ ]:
auto_ok = (
    auto["status"] == "completed"
    and auto_final["exit_code"] == 0
    and observed_green
    and writes >= 1
    and cycles >= 2
    and bool(auto.get("answer"))
    and auto_obs["tests_unchanged"]
    and not unauthorized
)

In [ ]:
assert auto_ok, f"autonomous run failed its criteria: {auto_obs}"
record_result("autonomous agent", auto_obs, auto_ok)

The trace shows the model using tools over several turns, changing source files, observing a green pytest result, and stopping with an answer. Python did not choose the model's actions, but it validated every call and enforced separate turn, tool-call, time, token, and cost budgets.

A final pytest invocation alone would not prove the agent saw success, so the criterion requires a passing `run_tests` observation inside the trajectory. Test hashes and the file allowlist add outcome checks. They still do not make the Python process safe against hostile code, which is why the sandbox limitation, revisited at the end, matters.

### Execution topologies

With the loop in hand, we can vary its shape. Once several calls are in play, the control flow takes a form worth naming. We walk five: a fixed chain, a router, a parallel fan-out, an orchestrator that plans its own workers, and a generate-and-check loop.

Before running them we write down, for every pattern, what we expect to happen. The ledger at the end grades observed against expected, with code where the outcome is objective and a model rubric only where quality is genuinely subjective. Declaring the criterion first keeps "it produced output" from masquerading as success.


#### Prompt chaining

A single call that must diagnose, edit, and review at once has no checkpoint between those jobs.

Prompt chaining splits them into an ordered sequence with a validated artifact passed between stages: diagnose, propose exact replacements, review the proposal. The gate that decides whether to apply is deterministic. Every proposed `old` string must exist in its file, and the patched result must still parse. The model's review runs too, but as a logged second opinion, because an application-critical decision should not hinge on a model returning the right boolean.

A chain runs one forward pass with no chance to react to a remaining failure. So we report the outcome as exactly one of `repaired`, `partial_repair`, `no_change`, or `invalid_patch`, and show it as its own column in the ledger. A reduced failure count is `partial_repair`, not a green suite, and we do not let it read as full success.


In [ ]:
PATCH_SCHEMA = strict(
    {
        "edits": {
            "type": "array",
            "items": strict(
                {
                    "path": {"enum": list(SOURCE_ALLOWLIST)},
                    "old": {"type": "string"},
                    "new": {"type": "string"},
                }
            ),
        },
        "rationale": {"type": "string"},
    }
)
REVIEW_SCHEMA = strict({"passed": {"type": "boolean"}, "feedback": {"type": "string"}})
start_experiment("prompt chaining")
diag, _ = call_structured(
    [
        {
            "role": "system",
            "content": "Diagnose every defect supported by the source and tests.",
        },
        {"role": "user", "content": json.dumps(gather_evidence(), sort_keys=True)},
    ],
    "prompt chaining",
    "diagnosis",
    DIAGNOSIS_SCHEMA,
)
patch, _ = call_structured(
    [
        {
            "role": "system",
            "content": "Propose exact old/new replacements fixing every diagnosed defect. Edit only pricing.py and orders.py.",
        },
        {
            "role": "user",
            "content": json.dumps({"diagnosis": diag, "files": snapshot()}, sort_keys=True),
        },
    ],
    "prompt chaining",
    "patch",
    PATCH_SCHEMA,
)

In [ ]:
review, _ = call_structured(
    [
        {
            "role": "system",
            "content": "Confirm every proposed old string exists in its file and the edits address the diagnosis. passed=false otherwise.",
        },
        {
            "role": "user",
            "content": json.dumps({"patch": patch, "files": snapshot()}, sort_keys=True),
        },
    ],
    "prompt chaining",
    "review",
    REVIEW_SCHEMA,
)
before = snapshot()
candidate = dict(before)

In [ ]:
errors = []
for edit in patch["edits"]:
    current = candidate[edit["path"]]
    if not edit["old"] or edit["old"] not in current:
        errors.append(f"{edit['path']}: old missing")
        continue
    updated = current.replace(edit["old"], edit["new"], 1)
    try:
        ast.parse(updated, filename=edit["path"])
    except SyntaxError as exc:
        errors.append(f"{edit['path']}: {exc.msg}")
        continue
    candidate[edit["path"]] = updated
patch_valid = bool(patch["edits"]) and not errors
if patch_valid:
    for path, content in candidate.items():
        if content != before[path]:
            write_file(path, content)
after = failure_count(run_tests())
if not patch_valid:
    chain_outcome = "invalid_patch"
elif after == 0:
    chain_outcome = "repaired"
elif after < 4:
    chain_outcome = "partial_repair"
else:
    chain_outcome = "no_change"
chain_obs = {
    "patch_valid": patch_valid,
    "model_review_passed": review["passed"],
    "failures_before": 4,
    "failures_after": after,
    "outcome": chain_outcome,
}
print("\nMEASURABLE RESULT:", json.dumps(chain_obs))
record_result(
    "prompt chaining",
    chain_obs,
    chain_outcome in ("repaired", "partial_repair"),
    "" if chain_outcome == "repaired" else f"one forward pass -> {chain_outcome} ({after} failing)",
)

The chain has three model stages, but Python owns the application gate. It simulates every proposed replacement against an in-memory candidate, rejects empty or missing match text, and parses the whole candidate before writing. The model review stays visible as a second opinion; it does not control the write.

We measure the result by the change in failing tests, reported as the outcome label. A single forward pass can cut the failure count without reaching a green suite, because no stage reacts to what remains. That is the tradeoff: the path is predictable, but a missed defect stays missed. The loop-based patterns later can revise after a new observation.

#### Routing

Sending every request through one pipeline wastes tools and context. A router asks the model for a single structured label, then Python validates and dispatches. The right version measures two things and keeps them apart: did the label match what we expected (route accuracy), and did the handler actually do its job (fulfilment)?

Reading a file or running the suite is not fulfilment. The bug-fix handler must make the requested failing test pass; reducing the overall failure count is insufficient. The test-authoring handler must produce a test whose assertion actually calls the requested function, checked on the AST, not a symbol that merely appears in the text. The explanation handler must cite symbols that really exist in the source.


In [ ]:
FIX_SCHEMA = PATCH_SCHEMA
TEST_SCHEMA = strict({"test_code": {"type": "string"}})
EXPLAIN_SCHEMA = strict(
    {
        "explanation": {"type": "string"},
        "symbols": {"type": "array", "items": {"type": "string"}},
    }
)
DEFINED_SYMBOLS = {
    n.name for p in SOURCE_ALLOWLIST for n in ast.walk(ast.parse(BASELINE_FILES[p])) if isinstance(n, (ast.FunctionDef, ast.ClassDef))
}


def asserts_call(code, name):
    tree = ast.parse(code)
    has_test = any(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_") for node in tree.body)
    call_in_assertion = any(
        isinstance(call, ast.Call) and getattr(call.func, "id", None) == name
        for node in ast.walk(tree)
        if isinstance(node, ast.Assert)
        for call in ast.walk(node.test)
    )
    return has_test and call_in_assertion

In [ ]:
def route_fix(request):
    patch, _ = call_structured(
        [
            {
                "role": "system",
                "content": "Fix the described defect with exact old/new edits to pricing.py or orders.py.",
            },
            {
                "role": "user",
                "content": json.dumps({"request": request, "files": snapshot()}, sort_keys=True),
            },
        ],
        "routing",
        "fix",
        FIX_SCHEMA,
    )
    edits = apply_edits(patch["edits"])
    hit = run_process(
        [
            sys.executable,
            "-m",
            "pytest",
            "-q",
            "-k",
            "test_percentage_discount",
            "--color=no",
        ],
        timeout=30,
        env=PYTEST_ENV,
    )
    return {
        "kind": "bug_fix",
        "fulfilled": hit["exit_code"] == 0,
        "rejected_edits": edits["rejected"],
    }

In [ ]:
def route_test(request):
    src, _ = call_structured(
        [
            {
                "role": "system",
                "content": "Write one pytest test function importing from pricing. The assertion itself must call percentage_discount, for example: assert percentage_discount(100, 0) == 100.",
            },
            {
                "role": "user",
                "content": json.dumps({"request": request, "files": snapshot()}, sort_keys=True),
            },
        ],
        "routing",
        "authoring",
        TEST_SCHEMA,
    )
    scratch = Path(tempfile.mkdtemp(prefix="route_"))
    for n in list(SOURCE_ALLOWLIST) + ["test_authored.py"]:
        (scratch / n).write_text(BASELINE_FILES.get(n, src["test_code"]), encoding="utf-8")
    try:
        ok_ast = asserts_call(src["test_code"], "percentage_discount")
    except SyntaxError:
        ok_ast = False
    ran = (
        ok_ast
        and subprocess.run(
            [sys.executable, "-m", "pytest", "-q", "--color=no"],
            cwd=scratch,
            text=True,
            capture_output=True,
            timeout=30,
            env=PYTEST_ENV,
        ).returncode
        == 0
    )
    shutil.rmtree(scratch, ignore_errors=True)
    return {"kind": "test_authoring", "fulfilled": ok_ast and ran}

In [ ]:
def route_explain(request):
    ex, _ = call_structured(
        [
            {
                "role": "system",
                "content": "Explain the relevant behaviour and list the source symbols you relied on.",
            },
            {
                "role": "user",
                "content": json.dumps({"request": request, "files": snapshot()}, sort_keys=True),
            },
        ],
        "routing",
        "explanation",
        EXPLAIN_SCHEMA,
    )
    grounded = bool(ex["symbols"]) and all(s in DEFINED_SYMBOLS for s in ex["symbols"])
    return {
        "kind": "code_explanation",
        "fulfilled": grounded and "can_cancel" in ex["symbols"] and "can_cancel" in ex["explanation"],
    }


HANDLERS = {
    "bug_fix": route_fix,
    "test_authoring": route_test,
    "code_explanation": route_explain,
}
ROUTE_SCHEMA = strict({"route": {"enum": list(HANDLERS)}})
route_cases = [
    (
        "The percentage discount function returns the wrong number; make its test pass.",
        "bug_fix",
    ),
    (
        "We need a test asserting a zero-percent discount returns the original price.",
        "test_authoring",
    ),
    (
        "Walk me through how order cancellation decides what may be cancelled.",
        "code_explanation",
    ),
]
rows = []

In [ ]:
for request, expected in route_cases:
    start_experiment("routing")
    decision, _ = call_structured(
        [
            {
                "role": "system",
                "content": "Route to exactly one workflow: bug_fix, test_authoring, or code_explanation.",
            },
            {"role": "user", "content": request},
        ],
        "routing",
        "route",
        ROUTE_SCHEMA,
    )
    out = HANDLERS[decision["route"]](request)
    rows.append(
        {
            "request": request[:42],
            "expected": expected,
            "route": decision["route"],
            "route_ok": decision["route"] == expected,
            "fulfilled": out["fulfilled"],
            "rejected_edits": len(out.get("rejected_edits", [])),
        }
    )
route_df = pd.DataFrame(rows)
print(route_df.to_string(index=False))

In [ ]:
route_acc, fulfil = route_df["route_ok"].mean(), route_df["fulfilled"].mean()
print(f"\nroute accuracy {route_acc:.0%} | fulfilment {fulfil:.0%}")
record_result(
    "routing",
    {"route_accuracy": round(route_acc, 2), "fulfilment": round(fulfil, 2)},
    route_acc == 1 and fulfil == 1,
    "" if route_acc == 1 and fulfil == 1 else "a route or handler failed",
)

Route accuracy and handler fulfilment are reported separately because either can fail. The bug-fix handler is graded on the specific test it was asked to fix. The test-authoring handler needs a `test_` function whose assertion calls `percentage_discount` and a passing pytest run in its own scratch fixture. The explanation handler needs exact source symbols, not loose substring matches.

Three examples are a demonstration, not a routing benchmark. A useful router evaluation needs a larger labelled set with ambiguous, adversarial, and out-of-distribution requests, plus confidence intervals and error analysis.

#### Parallelization

Independent reviews don't depend on each other's results, so they can run at the same time. Here we send three fixed review prompts (logic, coverage, validation) over the same repository, once in sequence and once through a thread pool, and compare wall time. This is plain fan-out over independent calls, not the same thing as forking an agent's context or spawning sub-agents, which carry extra state; those are separate ideas we are not demonstrating here.

The real trap is bookkeeping under concurrency. Each worker must get its own call id and read its own record from the return value, never from the shared log's last row, or two threads will scribble over each other's results. We assert that every id across the run is unique, keep the full responses in memory, and print only a compact table. One timing comparison is an illustration: the slowest worker sets the floor, and a rate limit can erase the gain.


In [ ]:
REVIEW_TASKS = {
    "logic": "Review calculation and state-transition logic.",
    "coverage": "Review what the tests cover and omit.",
    "validation": "Review missing input validation.",
}
start_experiment("parallelization")
review_ctx = json.dumps(gather_evidence(), sort_keys=True)


def review_worker(item):
    name, instruction = item
    rec = call_model(
        [
            {
                "role": "system",
                "content": instruction + " Reply in two sentences with file evidence.",
            },
            {"role": "user", "content": review_ctx},
        ],
        "parallelization",
    )
    return {
        "worker": name,
        "finding": (rec["content"] or "").strip().replace("\n", " ")[:70],
        "latency_s": rec["latency_s"],
        "cost": rec["cost"],
        "call_id": rec["call_id"],
    }


t0 = time.perf_counter()
sequential = [review_worker(it) for it in REVIEW_TASKS.items()]
seq_s = time.perf_counter() - t0
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    parallel = list(pool.map(review_worker, REVIEW_TASKS.items()))
par_s = time.perf_counter() - t0
all_ids = [r["call_id"] for r in sequential + parallel]
ids_unique = len(all_ids) == len(set(all_ids))
print(pd.DataFrame(parallel)[["worker", "finding", "latency_s", "cost"]].to_string(index=False))
par_obs = {
    "sequential_s": round(seq_s, 2),
    "parallel_s": round(par_s, 2),
    "speedup": round(seq_s / par_s, 2),
    "call_ids_unique": ids_unique,
}
print("\nMEASURABLE RESULT:", json.dumps(par_obs))
assert ids_unique, "duplicate call id under concurrency"
record_result("parallelization", par_obs, len(parallel) == 3 and ids_unique)

The three workers receive the same context and produce distinct call ids. Compare the measured sequential and parallel times above. The exact speedup depends on provider latency, so it should be read from the run rather than treated as a fixed result.


#### Orchestrator-workers

Some tasks are decomposed as they are understood, not from a fixed script. Here we use a constrained orchestrator: Python fixes the ownership boundary (one worker owns `pricing.py`, one owns `orders.py`), and the model writes each task's objective and does the analysis. That keeps the demonstration honest. We are not claiming the model invents an unconstrained decomposition, because the file ownership is prescribed. What the model contributes is the objective text and the findings.

We check that each source file is owned exactly once, that every worker's finding cites a function defined in its own file, and that the synthesis names both files in its repair steps. Weak decomposition would show up as a worker that cites nothing from its file, and the check would catch it.


In [ ]:
PLAN_SCHEMA = strict(
    {
        "tasks": {
            "type": "array",
            "minItems": 2,
            "maxItems": 2,
            "items": strict(
                {
                    "objective": {"type": "string"},
                    "owner_file": {"enum": list(SOURCE_ALLOWLIST)},
                }
            ),
        }
    }
)
SYNTH_SCHEMA = strict(
    {
        "prioritised_steps": {
            "type": "array",
            "minItems": 1,
            "items": {"type": "string"},
        },
        "files_addressed": {
            "type": "array",
            "minItems": 1,
            "items": {"enum": list(SOURCE_ALLOWLIST)},
        },
    }
)
start_experiment("orchestrator-workers")
orch_request = "Investigate why the pricing tests fail and how order cancellation is decided; give a prioritised repair plan spanning pricing.py and orders.py."
plan_instructions = "Write exactly two task objectives, one owning pricing.py and one owning orders.py via owner_file. Python fixes which file each task owns; you write the objective text."
plan, _ = call_structured(
    [
        {"role": "system", "content": plan_instructions},
        {"role": "user", "content": orch_request},
    ],
    "orchestrator-workers",
    "plan",
    PLAN_SCHEMA,
)
DEFINED_BY_FILE = {p: {n.name for n in ast.walk(ast.parse(BASELINE_FILES[p])) if isinstance(n, ast.FunctionDef)} for p in SOURCE_ALLOWLIST}

In [ ]:
def run_worker(task):
    f = task["owner_file"]
    rec = call_model(
        [
            {
                "role": "system",
                "content": "Complete the objective using only the supplied file. State findings and cite a function name from it.",
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "objective": task["objective"],
                        "file": f,
                        "content": BASELINE_FILES[f],
                    },
                    sort_keys=True,
                ),
            },
        ],
        "orchestrator-workers",
    )
    finding = (rec["content"] or "").strip()
    return {
        "objective": task["objective"],
        "owner_file": f,
        "finding": finding,
        "cites_file": any(s in finding for s in DEFINED_BY_FILE[f]),
    }

In [ ]:
workers = [run_worker(t) for t in plan["tasks"]]
synth, _ = call_structured(
    [
        {
            "role": "system",
            "content": "Synthesise the findings into one prioritised repair plan. Name pricing.py in its step and orders.py in its step; list both files_addressed.",
        },
        {
            "role": "user",
            "content": json.dumps({"request": orch_request, "workers": workers}, sort_keys=True),
        },
    ],
    "orchestrator-workers",
    "synthesis",
    SYNTH_SCHEMA,
)
required = set(SOURCE_ALLOWLIST)
owned = {t["owner_file"] for t in plan["tasks"]}
step_text = " ".join(synth["prioritised_steps"]).lower()
orch_obs = {
    "owner_files": sorted(owned),
    "each_source_owned_once": owned == required and len(plan["tasks"]) == 2,
    "workers_cite_files": all(w["cites_file"] for w in workers),
    "synthesis_names_both": required <= set(synth["files_addressed"]) and all(p in step_text for p in required),
}
print(
    "PLAN:",
    json.dumps([{"objective": t["objective"][:60], "owner_file": t["owner_file"]} for t in plan["tasks"]]),
)
print("\nMEASURABLE RESULT:", json.dumps(orch_obs))
orch_ok = orch_obs["each_source_owned_once"] and orch_obs["workers_cite_files"] and orch_obs["synthesis_names_both"]
record_result(
    "orchestrator-workers",
    orch_obs,
    orch_ok,
    "" if orch_ok else "plan, worker evidence, or synthesis failed a check",
)

Each source file is owned once, each worker cites a symbol from the file it was handed, and the synthesis names both files. These checks fit this deliberately file-scoped exercise. They do not prove that an arbitrary decomposition is optimal. General orchestrator evaluation needs task-specific coverage rules, dependency checks, and human-calibrated judgements about redundant or missing work.

#### Evaluator-optimizer

The optimizer proposes a patch and receives feedback over several iterations. Pytest is the hard correctness gate. A separate call grades evidence, scope, validation, and explanation quality. Because the notebook uses the same configured model to generate and to grade, this is a separate evaluator call, not an independent evaluator model.

The model reports only the four component flags. Python computes the aggregate pass from them, so the model cannot wave a candidate through by asserting its own overall score. The loop stops when pytest passes and the recomputed rubric passes, or when the iteration budget runs out. The first candidate is a labelled controlled seed that fixes only `percentage_discount`, which guarantees at least one real revision without pretending the model produced the partial starting point.


In [ ]:
RUBRIC_SCHEMA = strict(
    {
        **{
            name: {"type": "boolean"}
            for name in [
                "evidence_coverage",
                "scope_ok",
                "validation_complete",
                "explanation_quality",
            ]
        },
        "feedback": {"type": "string"},
    }
)

start_experiment("evaluator-optimizer")
CONTROLLED_SEED = """def percentage_discount(price, percent):
    return round(price * (1 - percent / 100), 2)


def shipping_cost(subtotal):
    return 0.0 if subtotal > 50 else 5.0
"""
write_file("pricing.py", CONTROLLED_SEED)
remaining = failure_count(run_tests())
print("controlled seed applied (percentage_discount only); remaining failures:", remaining)

In [ ]:
def optimizer_iteration(iteration, feedback):
    patch, _ = call_structured(
        [
            {
                "role": "system",
                "content": (
                    "Propose exact old/new edits to pricing.py or orders.py "
                    "fixing the remaining failures. Follow the feedback. "
                    "Do not edit tests."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "iteration": iteration,
                        "feedback": feedback,
                        "files": snapshot(),
                        "tests": run_tests()["stdout"],
                    },
                    sort_keys=True,
                ),
            },
        ],
        "evaluator-optimizer",
        "patch",
        PATCH_SCHEMA,
    )
    edits = apply_edits(patch["edits"])
    tests = run_tests()
    tests_pass = tests["exit_code"] == 0

    rubric, _ = call_structured(
        [
            {
                "role": "system",
                "content": (
                    "Grade evidence_coverage, scope_ok, validation_complete, and explanation_quality. Report each flag, with no aggregate."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "patch": patch,
                        "files": snapshot(),
                        "pytest_tail": tests["stdout"][-400:],
                    },
                    sort_keys=True,
                ),
            },
        ],
        "evaluator-optimizer",
        "rubric",
        RUBRIC_SCHEMA,
    )
    rubric_fields = [
        "evidence_coverage",
        "scope_ok",
        "validation_complete",
        "explanation_quality",
    ]
    rubric_pass = all(rubric[field] for field in rubric_fields)
    row = {
        "iteration": iteration,
        "tests_pass": tests_pass,
        "rubric_pass": rubric_pass,
        "rejected_edits": len(edits["rejected"]),
    }
    next_feedback = rubric["feedback"] + "\nPytest tail:\n" + tests["stdout"][-300:]
    return row, next_feedback

In [ ]:
feedback = "Start from the seed. Fix the remaining failures with exact edits and validate inputs."
history = []

for iteration in range(1, 5):
    result, feedback = optimizer_iteration(iteration, feedback)
    history.append(result)
    print(
        f"iter {iteration}: tests_pass={result['tests_pass']} rubric_pass={result['rubric_pass']} rejected_edits={result['rejected_edits']}"
    )
    if result["tests_pass"] and result["rubric_pass"]:
        break

In [ ]:
eo_obs = {
    "iterations": len(history),
    "final_tests_pass": history[-1]["tests_pass"],
    "final_rubric_pass": history[-1]["rubric_pass"],
}
print("\nMEASURABLE RESULT:", json.dumps(eo_obs))
record_result(
    "evaluator-optimizer",
    eo_obs,
    history[-1]["tests_pass"] and history[-1]["rubric_pass"],
    "" if history[-1]["tests_pass"] else "budget exhausted before tests passed",
)

The two gates answer different questions. Pytest checks behaviour encoded in tests. The rubric checks properties the tests do not fully capture. Python, not the evaluator, decides whether the component flags amount to a pass.

This remains a toy evaluation. The rubric has not been calibrated against human reviewers, and the same model family proposes and grades the patch. A production evaluation should compare graders with human labels, test for grader manipulation, and run repeated trials over a task set.

### Manager versus handoff

With several specialists in play, one question decides the architecture: who owns the final answer? A manager keeps ownership and treats specialists as tools it calls. A handoff transfers ownership outright, so a specialist runs the rest of the task and speaks last. The OpenAI Agents SDK draws exactly this line, between agents-as-tools, where a manager stays in control, and handoffs, where control moves across.


#### Manager with specialists

The manager has three specialist tools with different instructions: debugging, testing, and code review. It should choose a non-empty subset and write the final recommendation itself. The runtime permits at most two successful consultations and rejects a repeated role. The criterion is strict about repetition: if the manager calls the same specialist twice, even when the second call is rejected, the run fails the criterion. The manager gets enough turns to consult and then answer on its own, so no separate forced synthesis call is needed.

A single-agent baseline receives the same repository context. The notebook compares cost, but it does not claim the manager is better from one example. Establishing a quality gain would need repeated, blinded comparison across a task set.


In [ ]:
SPECIALISTS = {
    "debugging": "Identify root causes and cite the exact source lines.",
    "testing": "Interpret the pytest failures and the missing coverage. Do not change tests.",
    "code_review": "Judge correctness, validation, and regression risk.",
}
start_experiment("manager-specialists")
mgr_ctx = json.dumps(gather_evidence(), sort_keys=True)
consulted = []


def consult(role, question):
    if role in consulted:
        raise ValueError(f"{role} was already consulted")
    if len(consulted) >= 2:
        raise ValueError("consultation limit reached")
    consulted.append(role)
    rec = call_model(
        [
            {"role": "system", "content": SPECIALISTS[role]},
            {
                "role": "user",
                "content": json.dumps({"question": question, "repository": mgr_ctx}, sort_keys=True),
            },
        ],
        "manager-specialists",
    )
    directive = (
        " You now have two specialist replies. Do not call any more tools; write your final recommendation." if len(consulted) >= 2 else ""
    )
    return {
        "specialist": role,
        "answer": (rec["content"] or "")[:400],
        "next": directive,
    }


for role in SPECIALISTS:
    TOOL_SPECS[f"consult_{role}"] = (
        f"Consult the {role.replace('_', ' ')} specialist once.",
        {"question": {"type": "string"}},
        ["question"],
    )
    TOOL_REGISTRY[f"consult_{role}"] = (lambda r: lambda question: consult(r, question))(role)

In [ ]:
manager = run_agent(
    [
        {
            "role": "system",
            "content": (
                "You lead a repository review. Consult at most two specialists to inform your decision, each exactly once. "
                "Never call the same specialist twice. Once you have two replies, stop calling tools and write the final recommendation and repair order yourself."
            ),
        },
        {
            "role": "user",
            "content": "Explain why mini_project fails and recommend a repair order.",
        },
    ],
    [f"consult_{r}" for r in SPECIALISTS],
    "manager-specialists",
    budget=4,
    max_tool_calls=2,
)

In [ ]:
attempts = [e["action"] for e in manager["trace"] if e["action"].startswith("consult_")]
successful = [e["action"] for e in manager["trace"] if e["action"].startswith("consult_") and e.get("ok")]
rejected = [e["action"] for e in manager["trace"] if e["action"].startswith("consult_") and e.get("ok") is False]
repeated_role = len(attempts) != len(set(attempts))
baseline = call_model(
    [
        {
            "role": "system",
            "content": "You are one agent. Explain why mini_project fails and recommend a repair order.",
        },
        {"role": "user", "content": mgr_ctx},
    ],
    "manager-baseline",
)
mgr_obs = {
    "consulted": sorted(consulted),
    "successful": successful,
    "rejected": rejected,
    "repeated_role": repeated_role,
    "bounded_subset": 1 <= len(consulted) <= 2,
    "manager_owns_answer": bool(manager.get("answer")),
    "manager_cost": summarize_calls("manager-specialists")["cost"],
    "single_agent_cost": baseline["cost"],
}
print("\nMEASURABLE RESULT:", json.dumps(mgr_obs))
mgr_ok = (not repeated_role) and mgr_obs["bounded_subset"] and mgr_obs["manager_owns_answer"]
record_result(
    "manager-specialists",
    mgr_obs,
    mgr_ok,
    "" if mgr_ok else "manager repeated a consult, exceeded bounds, or gave no answer",
)

The manager kept ownership of the user-facing answer, consulted a bounded and non-repeated subset, and wrote the recommendation itself. The baseline shows the extra cost of delegation. One run cannot tell us whether that spend bought quality, so the notebook reports the comparison without turning it into a performance claim.

#### Handoff

Sometimes triage should step aside and let one specialist own the response. A handoff does that. A triage call picks the owner, then a fresh agent run starts under that specialist's instructions and tool boundary and produces the final answer. The specialist is handed the file listing along with the request, so its first action lands on a real path rather than a guess, and that first-action validity is part of the pass condition. The trace records the transfer: the triage decision, the change of active instructions, the new tool set, the context carried across, and the closing answer. The contrast with the manager is the whole point. There, control never left the manager; here it moves, and the specialist speaks last.


In [ ]:
HANDOFF_SCHEMA = strict({"specialist": {"enum": list(SPECIALISTS)}, "reason": {"type": "string"}})
HANDOFF_TOOLS = {
    "debugging": ["list_files", "read_file", "search_code", "run_tests"],
    "testing": ["list_files", "read_file", "run_tests"],
    "code_review": ["list_files", "read_file", "search_code"],
}
start_experiment("handoff")
handoff_request = "Find the root causes of the failing discount and shipping tests and give the final diagnosis."
triage_instructions = "Choose the one specialist that should own the response. Do not answer the task yourself."
triage, _ = call_structured(
    [
        {"role": "system", "content": triage_instructions},
        {"role": "user", "content": handoff_request},
    ],
    "handoff",
    "triage",
    HANDOFF_SCHEMA,
)
owner = triage["specialist"]
active_tools = HANDOFF_TOOLS[owner]
owner_instructions = f"You now own the response. {SPECIALISTS[owner]} Use the read-only tools on the listed files, then give the final diagnosis naming percentage_discount and shipping_cost."
ACTIVE_AGENT.append(owner + "_owner")
run = run_agent(
    [
        {"role": "system", "content": owner_instructions},
        {
            "role": "user",
            "content": json.dumps(
                {
                    "request": handoff_request,
                    "triage_reason": triage["reason"],
                    "available_files": list_files(),
                }
            ),
        },
    ],
    active_tools,
    "handoff",
    budget=8,
)
ACTIVE_AGENT.pop()
answer = run.get("answer") or ""
actions = [r["action"] for r in run["trace"] if r["action"] != "stop"]
first_action_valid = bool(run["trace"]) and run["trace"][0].get("ok") is True
grounded = all(s in answer for s in ["percentage_discount", "shipping_cost"])

In [ ]:
handoff_obs = {
    "triage_specialist": owner,
    "instructions_changed": owner_instructions != triage_instructions,
    "active_tools": active_tools,
    "tool_actions": actions,
    "first_action_valid": first_action_valid,
    "specialist_owns_answer": bool(answer),
    "diagnosis_grounded": grounded,
}
print("\nHANDOFF TRACE:")
show_trace(run["trace"])
print("\nMEASURABLE RESULT:", json.dumps(handoff_obs))
handoff_ok = handoff_obs["instructions_changed"] and bool(actions) and first_action_valid and bool(answer) and grounded

In [ ]:
assert handoff_ok
record_result("handoff", handoff_obs, handoff_ok)

Ownership moved from triage to the selected specialist. The pass condition requires changed instructions, a valid first action, at least one specialist tool action, and a final diagnosis that names both requested functions. That is stronger than treating any non-empty answer as a successful handoff.

The example still tests one narrow request. A full handoff evaluation would include incorrect routing, missing context, rejected transfers, return-to-manager paths, and conversations spanning several user turns.

## Diagnostics & Robustness


### Runtime boundaries and approval

The model never receives direct filesystem or process authority. Python validates JSON arguments, resolves paths inside `mini_project`, restricts writes to `pricing.py` and `orders.py`, disables third-party pytest plugin autoloading, gives subprocesses timeouts, and applies several runtime budgets.

These are application controls, not an operating-system sandbox. Allowed source files execute inside pytest and therefore inherit the Python process's authority. A production coding agent needs a disposable container or stronger sandbox, restricted network egress, limited credentials, and resource quotas. The check below exercises a representative slice of the write boundary: one allowed write, then a protected test, a new file, a path escape, and a symlink swap.


In [ ]:
def attempt_write(path, content="x = 1\n"):
    try:
        write_file(path, content)
        return {"attempt": path, "rejected": False}
    except (PermissionError, ValueError) as exc:
        return {"attempt": path, "rejected": True, "error": type(exc).__name__}


start_experiment("runtime controls")
allowed = write_file("pricing.py", AUTO_FIXED["pricing.py"])
rows = [attempt_write(p) for p in ["tests/test_pricing.py", "helper.py", "../escape.py"]]
try:
    outside = WORKSPACE / "outside.py"
    outside.write_text("z = 3\n", encoding="utf-8")
    link = PROJECT_ROOT / "pricing.py"
    link.unlink()
    os.symlink(outside, link)
    rows.append(
        {
            "attempt": "symlink swap",
            "rejected": attempt_write("pricing.py", "x = 0\n")["rejected"],
            "error": "PermissionError",
        }
    )
except OSError:
    rows.append({"attempt": "symlink swap", "rejected": "skipped_no_symlink_priv"})
print("allowed write pricing.py ->", allowed)
print(pd.DataFrame(rows).to_string(index=False))
all_rejected = all(r["rejected"] is True or r.get("rejected") == "skipped_no_symlink_priv" for r in rows)
assert allowed["changed"] and all_rejected, "a boundary check failed"
print("\nallowed write succeeded; protected-test, new-file, escape, and symlink writes rejected")
record_result(
    "runtime controls",
    {
        "allowed_write": allowed["changed"],
        "rejected": [r["attempt"] for r in rows],
        "all_rejected": all_rejected,
    },
    allowed["changed"] and all_rejected,
)

These cases demonstrate the boundaries we actually check: an allowed source write goes through, while a protected test, a new file, a path escaping the project, and a symlink swap are all rejected. The symlink case matters because writing through a link would land bytes outside the project even though the name looked local. This is a representative slice, not proof that every possible route is closed, and it is still not a sandbox.

Approval is the other control, for the one action with real consequences: a commit. The model can propose it, but proposing is all it can do. The notebook then supplies the decision automatically, which stands in for a human approval step. We serialise the exact approved state (a hash of the diff, the base commit it applies to, and an idempotency key) and require that separate decision to act.

In [ ]:
GIT_ENV = {
    **os.environ,
    "GIT_AUTHOR_DATE": "2020-01-01T00:00:00 +0000",
    "GIT_COMMITTER_DATE": "2020-01-01T00:00:00 +0000",
}
RUNTIME_DIR = PROJECT_ROOT / ".agent_runtime"
COMPLETED = RUNTIME_DIR / "completed.json"


def git_head():
    return run_process(["git", "rev-parse", "HEAD"])["stdout"].strip()


def git_init():
    for cmd in (
        ["git", "init", "-q"],
        ["git", "config", "user.email", "s@example.com"],
        ["git", "config", "user.name", "Student"],
        ["git", "add", "."],
    ):
        run_process(cmd)
    run_process(["git", "commit", "-qm", "baseline with intentional defects"], env=GIT_ENV)


def current_diff():
    return run_process(["git", "diff", "--", *SOURCE_ALLOWLIST])["stdout"]


def propose_commit(message):
    diff = current_diff()
    if not diff.strip():
        return {"status": "no_changes"}
    return {
        "status": "pending_approval",
        "message": message,
        "diff_hash": hashlib.sha256(diff.encode()).hexdigest()[:16],
        "base_commit": git_head(),
        "idempotency_key": hashlib.sha256((message + diff).encode()).hexdigest()[:16],
    }

In [ ]:
def resume_commit(state):
    done = json.loads(COMPLETED.read_text(encoding="utf-8"))
    if state["idempotency_key"] in done:
        return {"status": "already_done", "committed": False, "duplicate": True}
    if git_head() != state["base_commit"]:
        return {"status": "rejected_stale", "committed": False}
    if hashlib.sha256(current_diff().encode()).hexdigest()[:16] != state["diff_hash"]:
        return {"status": "rejected_modified", "committed": False}
    if run_tests()["exit_code"] != 0:
        return {"status": "rejected_failing", "committed": False}
    if run_process(["git", "add", *SOURCE_ALLOWLIST])["exit_code"] != 0:
        return {"status": "staging_failed", "committed": False}
    commit = run_process(["git", "commit", "-m", state["message"]], env=GIT_ENV)
    new_head = git_head()
    if commit["exit_code"] != 0 or new_head == state["base_commit"]:
        return {"status": "commit_failed", "committed": False}
    done.append(state["idempotency_key"])
    COMPLETED.write_text(json.dumps(done), encoding="utf-8")
    return {
        "status": "committed",
        "committed": True,
        "duplicate": False,
        "new_head": new_head,
    }

In [ ]:
def prepare_repo():
    start_experiment("approval")
    git_init()
    write_file("pricing.py", AUTO_FIXED["pricing.py"])
    write_file("orders.py", AUTO_FIXED["orders.py"])
    assert run_tests()["exit_code"] == 0, "approval demo needs a passing repository"
    RUNTIME_DIR.mkdir(exist_ok=True)
    COMPLETED.write_text("[]", encoding="utf-8")


prepare_repo()
proposal = call_model(
    [
        {
            "role": "system",
            "content": "The tests pass. Propose one concise git commit message for these source changes.",
        },
        {"role": "user", "content": current_diff()[:2000]},
    ],
    "approval",
)
pending = propose_commit((proposal["content"] or "Fix pricing and order defects").strip().splitlines()[0][:72])
approved = resume_commit(pending)
replay = resume_commit(pending)
print("APPROVE:", json.dumps(approved))
print("IDEMPOTENT REPLAY:", json.dumps(replay))
prepare_repo()
tamper_pending = propose_commit("second commit")
safe_path("orders.py").write_text(AUTO_FIXED["orders.py"] + "\n# drifted after approval\n", encoding="utf-8")
tampered = resume_commit(tamper_pending)
print("MODIFIED-DIFF REJECTION:", json.dumps(tampered))
approval_obs = {
    "approved": approved["committed"],
    "replay_idempotent": replay.get("duplicate", False),
    "modified_diff_rejected": tampered["status"] == "rejected_modified",
}
assert all(approval_obs.values()), approval_obs
record_result("approval", approval_obs, all(approval_obs.values()))

The approval state binds the decision to the exact diff, its base commit, and an idempotency key. The approved path verifies that staging succeeded, the commit command returned zero, and Git `HEAD` advanced before it records completion. A replay returns `already_done` rather than creating a second commit, and a modified working tree fails the diff-hash check.

This is a simulated human decision inside one process, which demonstrates replay protection, not a distributed exactly-once guarantee. Durable systems need transactional state or an operation ledger that survives a crash between the external action and the completion record.

### Run ledger and evaluation

Each pattern declares its expected behaviour before it runs. Objective properties use code checks: pytest status, file hashes, exact symbols, bounded specialist calls, single-file ownership, Git state, or rejected writes. Model grading is reserved for subjective answer quality, and Python recomputes any aggregate from component flags. There is no extra model grader here; the evaluator-optimizer section already showed model-based grading under a hard test gate.

The final table is a run ledger, not a reliability estimate. A `criterion_met` value means one execution met the criterion this notebook declared for that pattern, and the chaining row carries its outcome label so a partial repair never reads as a green suite. Every pattern the notebook demonstrated appears as a row, including the plain-call negative control and the runtime-control checks.


In [ ]:
LEDGER = [
    "plain call",
    "augmented LLM",
    "workflow",
    "agent",
    "autonomous agent",
    "prompt chaining",
    "routing",
    "parallelization",
    "orchestrator-workers",
    "evaluator-optimizer",
    "manager-specialists",
    "handoff",
    "runtime controls",
    "approval",
]
assert set(LEDGER) == set(EXPECTED), "ledger must list every registered pattern"
missing = [p for p in LEDGER if p not in RESULTS]
assert not missing, f"unrecorded demonstrations: {missing}"


def outcome_of(p):
    obs = RESULTS[p]["observed"]
    return obs.get("outcome", "") if isinstance(obs, dict) else ""


audit = pd.DataFrame([{**RESULTS[p], "outcome": outcome_of(p)} for p in LEDGER])[
    ["pattern", "criterion_met", "outcome", "calls", "tokens", "cost", "note"]
]
print("PATTERN LEDGER (every demonstrated concept):")
print(audit.to_string(index=False))
totals = {
    "live_calls": len(CALL_LOG),
    "errors": [r["error"] for r in CALL_LOG if r["error"]],
    "total_tokens": sum((r["tokens"] or 0) for r in CALL_LOG),
    "total_cost_usd": round(sum((r["cost"] or 0) for r in CALL_LOG), 5),
    "unique_call_ids": len({r["call_id"] for r in CALL_LOG}) == len(CALL_LOG),
    "models": sorted({r["actual_model"] for r in CALL_LOG if r.get("actual_model")}),
}
print("\nRUN TOTALS:", json.dumps(totals, indent=2))
met = int(audit["criterion_met"].sum())
print(f"\n{met}/{len(LEDGER)} demonstrations met their declared criterion this run")
for p in LEDGER:
    if not RESULTS[p]["criterion_met"]:
        print("UNMET:", p, "->", RESULTS[p]["note"])
assert not totals["errors"], f"live errors present: {totals['errors']}"

In [ ]:
assert totals["unique_call_ids"], "duplicate call id in the full log"
assert RESULTS["plain call"]["criterion_met"] and RESULTS["autonomous agent"]["criterion_met"]

Read the table as a ledger. Each row records whether this run met a criterion declared in code, alongside calls, tokens, and cost. The plain call is a negative control: it "passes" by leaving the environment unchanged. Every other row concerns one model, one toy repository, and one trial. Model sampling, provider routing, and small prompt changes can all move the result.

## Discussion & Limitations

The repository is tiny, its context fits in memory, its tools are local, and pytest supplies an unusually clean success signal. Passing tests can still miss unstated requirements, and one run cannot establish that a pattern is reliable or cost-effective.

Production systems add the difficult parts omitted here: retrieval and context management, prompt-injection defence, retries and recovery, evaluation over task suites, credential isolation, and deployment. The patterns in this notebook are a vocabulary for reasoning about those systems, not a production blueprint.


## Conclusion

Agency is about control over action selection. A model handed extra context is an augmented call. A fixed sequence of model calls is a workflow. A model that chooses tools after observing prior results is acting as an agent, within the limits its runtime imposes. That last clause matters: the useful systems mix model choice with deterministic controls.


## Reproducibility & Commands
Install the runtime dependencies in an isolated environment:

```powershell
python -m pip install pandas python-dotenv jsonschema pytest jupyter nbconvert
```

Set `OPENROUTER_API_KEY` in `.env`, ensure Git is available on `PATH`, and execute the notebook from top to bottom:

```powershell
conda run -n quant jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 agents_how_they_work.ipynb
```

The run makes paid API requests. Its final ledger reports the observed call count, tokens, and cost. A successful run is evidence for that execution only.

Export HTML, PDF, and DOCX versions with:

```powershell
conda run -n quant python scripts/export_notebook.py --notebook agents_how_they_work.ipynb --output-dir exports
```

Pandoc is required for DOCX export. PDF export also needs a TeX engine such as `xelatex`.

The notebook is available on [GitHub](https://github.com/adamd1985/quant_research/blob/main/agents_how_they_work.ipynb) and [Kaggle](https://www.kaggle.com/code/addarm/how-agents-work).


## References

1. Anthropic (2024). *Building Effective Agents*. https://www.anthropic.com/engineering/building-effective-agents
2. Anthropic (2026). *Demystifying Evals for AI Agents*. https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents
3. OpenAI. *Agents SDK: Orchestrating Multiple Agents*. https://openai.github.io/openai-agents-python/multi_agent/
4. Yao, S. et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models*. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
5. OpenRouter. *Structured Outputs*. https://openrouter.ai/docs/guides/features/structured-outputs
6. OpenRouter. *Provider Routing*. https://openrouter.ai/docs/guides/routing/provider-selection
7. pytest. *Writing Plugins*. https://docs.pytest.org/en/stable/how-to/writing_plugins.html
